# End-to-End DS project in Databricks
## - Data Ingestion (UnityCatalog)
## - SQL
## - Spark Loading
## - Visualization
## - FeatureStore
## - Model
## - Log, Register
## - Deploy
## - Inference
## - Check

In [0]:
%sql
SELECT CURRENT_METASTORE();

# SQL

In [0]:
%sql
-- SELECT * FROM hive_metastore.default.loans_prediction LIMIT 10;
SELECT * FROM training.loans_prediction.train LIMIT 10;

# Load data from UnityCatalog

## Visualization tool

In [0]:
# df = spark.sql("SELECT * FROM hive_metastore.default.loans_prediction")  # from legacy hivemetastore, via custom sql
df = spark.table("training.loans_prediction.train") # unity catalog is disabled when using "no isolation shared" clusters
display(df)

Databricks data profile. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

# Feature Store

## Split train-test and store split as a column

In [0]:
from pyspark.sql.functions import lit, when
from pyspark.sql import DataFrame

# Split the dataframe into train and test sets
train_df, test_df = df.randomSplit([0.8, 0.2], seed=22)

# Add a column specifying if the row belongs to train_set or test_set
train_df = train_df.withColumn("set_type", lit("train"))
test_df = test_df.withColumn("set_type", lit("test"))

# Union the train and test dataframes to keep the original dataframe full
full_df_with_set_type = train_df.union(test_df)

display(full_df_with_set_type)

## Create Features

In [0]:
from pyspark.sql.functions import log

# Create feature as log(income)
full_df_with_set_type = full_df_with_set_type.withColumn("log_person_income", log("person_income"))

full_df_with_set_type.display()

## Store into feature store

### [FAIL] Try to create new table with few features, then create a feature table associated, and populated it with my features.. 
When populating, seems ot be an issue with the types, 
Actually, the fe.create_table prolly doesnt do anything, so the table has the schema created via the sql, and that why i cant push to it since the spark df has a different type for its cols

In [0]:
# %sql
# CREATE TABLE training.loans_prediction.train_features (
#   id int NOT NULL,
#   person_age INT, 
#   loan_percent_income FLOAT, 
#   loan_status BOOLEAN,
#   CONSTRAINT loans_prediction_train_features_pk PRIMARY KEY (id)
# );

In [0]:
# from databricks.feature_engineering import FeatureEngineeringClient
# from databricks.feature_store import FeatureStoreClient

# # Define our "features" - all the columns we want to store in the FeatureStore
# feature_cols = [
#   "id", 
#   "log_person_income", 
#   "person_age", 
#   "loan_percent_income", 
#   "loan_status", 
#   "set_type",
# ]

# # Define the feature table name - linked to the "training" catalog
# # feature_table_name = "training.loans_prediction.train"

# # # Add primary key constraint to the table
# # sql_cmd = f"ALTER TABLE {feature_table_name} ADD CONSTRAINT {feature_table_name.split('.')[-2]}_{feature_table_name.split('.')[-1]}_pk PRIMARY KEY (id)"
# # spark.sql(sql_cmd)


# feature_table_name = "training.loans_prediction.train_features"


# # Initialize the Feature Eng client
# fe = FeatureEngineeringClient()

# # Create or update the feature table in Unitycatalog
# fe.create_table(
#     name=feature_table_name,
#     primary_keys=["id"],
#     df=full_df_with_set_type.select(*feature_cols),
#     description="Feature table for loans prediction"
# )

### Create Feature table in UC, populate it

In [0]:
from databricks.feature_engineering import FeatureEngineeringClient
from databricks.feature_store import FeatureStoreClient

# Define our "features" - all the columns we want to store in the FeatureStore
feature_cols = [
  "id", 
  "log_person_income", 
  "person_age", 
  "loan_percent_income", 
  "loan_status", 
  "set_type",
]

# Define the feature table name - linked to the "training" catalog
feature_table_name = "training.loans_prediction.train_features" + "_v1"

fe = FeatureEngineeringClient()

# delete if exists
try:
  fe.get_table(name=feature_table_name)
  fe.delete_table(feature_table_name)
except Exception:
  # table does not exist
  pass

fe.create_table(
    name=feature_table_name,
    primary_keys=["id"],
    df=full_df_with_set_type.select(*feature_cols),
    description="Feature table for loans prediction",
    tags={'version': '1'}
)

# Initialize the Feature Store client
fs = FeatureStoreClient()

# Write the new feature to the feature store
fs.write_table(
    name=feature_table_name,
    df=full_df_with_set_type.select(*feature_cols),
    # mode="merge",
    mode="overwrite"
)

## Load Train and Test Data from Feature Store

In [0]:
from pyspark.sql.functions import col

# load from feature store
# feature_table_name = "hive_metastore.default.loans_prediction_data"
feature_table_name = "training.loans_prediction.train_features" + "_v1"
train_df = fs.read_table(name=feature_table_name).filter(col("set_type") == "train")
test_df = fs.read_table(name=feature_table_name).filter(col("set_type") == "test")

# switch to pandas
train_df_pandas = train_df.toPandas()
test_df_pandas = test_df.toPandas()

# create X and y
num_samples = -1
X_train = train_df_pandas[["person_age", "log_person_income", "loan_percent_income"]][:num_samples].values
y_train = train_df_pandas["loan_status"][:num_samples].values

X_test = test_df_pandas[["person_age", "log_person_income", "loan_percent_income"]].values
y_test = test_df_pandas["loan_status"].values

# Train Model and log with MLFlow

In [0]:
#  If trying to access models in the Unity Catalog, make sure to install version 2.4.1 or above of the MLflow Python client, then configure the MLflow client to access models in Unity Catalog via mlflow.set_registry_uri('databricks-uc')

import mlflow
print(mlflow.__version__)
print(f'Current model registry [{mlflow.get_registry_uri()}]')
mlflow.set_registry_uri('databricks-uc')
print(f'Current model registry [{mlflow.get_registry_uri()}]')

In [0]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score)

n_estimators =100
seed = 22
# seed = 42

with mlflow.start_run() as run:
    # Log parameters
    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("random_state", seed)
    
    # create model and train
    rf = RandomForestClassifier(n_estimators=n_estimators, oob_score=True, random_state=seed)
    rf.fit(X_train, y_train)

    # evaluate
    predicted = rf.predict(X_test)
    accuracy = accuracy_score(y_test, predicted)
    f1 = f1_score(y_test, predicted, average='weighted')
    p = precision_score(y_test, predicted)
    r = recall_score(y_test, predicted)

    print(f'Mean accuracy score: {accuracy:.3}')
    print(f'Weighted F1 score: {f1:.3}')
    print(f'Binary Precision score: {p:.3}')
    print(f'Binary Recall score: {r:.3}')
    print(f'F1 binary score: {(2/(1/r + 1/p)):.3}')

    # log metrics
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("precision", p)
    mlflow.log_metric("recall", r)

    # log model as an sklearn model - provide sample to auto-set the model signature
    mlflow.sklearn.log_model(rf, "model", input_example=X_train[0].reshape(1, -1))
    
    # Store run uri for later
    run_uri = f"runs:/{run.info.run_id}/model"

## Register model

In [0]:
# register model to Unity Catalog
model_name = "training.loans_prediction.RandomForestClassifierModel"
model_version = mlflow.register_model(run_uri, model_name)

# BONUS
# FULL SPARK EXAMPLE
# Note VecAssembler does not work in serverless, High-concurrency cluster.
import os
from databricks.feature_store import FeatureStoreClient
import mlflow
from pyspark.ml import Pipeline
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import (BinaryClassificationEvaluator, MulticlassClassificationEvaluator)
from pyspark.ml.feature import VectorAssembler
from pyspark.sql.functions import col

os.environ["DISABLE_MLFLOWDBFS"] = "true"
mlflow.set_registry_uri('databricks-uc')

fs = FeatureStoreClient()

# Define feature columns and label column
feature_cols = ["person_age", "log_person_income", "loan_percent_income"]
label_col = "loan_status"
num_trees = 99
max_depth = 25

# Load data
feature_table_name = "training.loans_prediction.train_features" + "_v1"
train_df = fs.read_table(name=feature_table_name).filter(col("set_type") == "train")
test_df = fs.read_table(name=feature_table_name).filter(col("set_type") == "test")

with mlflow.start_run() as run:
    # Log parameters
    mlflow.log_param("num_trees", num_trees)
    mlflow.log_param("max_depth", max_depth)
    
    # Assemble features
    assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

    # Define Random Forest model
    rf = RandomForestClassifier(numTrees=num_trees, maxDepth=max_depth, labelCol=label_col, featuresCol="features")

    # Create a pipeline
    pipeline = Pipeline(stages=[assembler, rf])

    # Fit the model
    model = pipeline.fit(train_df)

    # Make predictions
    predictions = model.transform(test_df)

    # Score
    # Binary classification metrics
    binary_evaluator = BinaryClassificationEvaluator(labelCol=label_col, rawPredictionCol="rawPrediction")
    auroc = binary_evaluator.evaluate(predictions, {binary_evaluator.metricName: "areaUnderROC"})
    aupr = binary_evaluator.evaluate(predictions, {binary_evaluator.metricName: "areaUnderPR"})

    multiclass_evaluator = MulticlassClassificationEvaluator(labelCol=label_col, predictionCol="prediction")
    accuracy = multiclass_evaluator.evaluate(predictions, {multiclass_evaluator.metricName: "accuracy"})
    p = multiclass_evaluator.evaluate(predictions, {multiclass_evaluator.metricName: "weightedPrecision"})
    r = multiclass_evaluator.evaluate(predictions, {multiclass_evaluator.metricName: "weightedRecall"})
    f1 = (2/(1/r + 1/p))

    print(f'Mean accuracy score: {accuracy:.3}')
    print(f'AUROC score: {auroc:.3}')
    print(f'AUPR score: {aupr:.3}')
    print(f'Weighted Precision score: {p:.3}')
    print(f'Weighted Recall score: {r:.3}')
    print(f'Weighted F1 score: {f1:.3}')

    # log metrics
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("auroc", auroc)
    mlflow.log_metric("aupr", aupr)
    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("precision", p)
    mlflow.log_metric("recall", r)

    # log Spark x MLlib model
    mlflow.spark.log_model(model, "model", input_example=train_df.select(*feature_cols).limit(1).toPandas())

# register model
model_uri = f"runs:/{run.info.run_id}/model"
model_name = "training.loans_prediction.SparkRandomForestClassifierModel"
model_version = mlflow.register_model(model_uri, model_name)

# Inference

## Batch Inference

In [0]:
# Allow Scaling using Spark UDF

import mlflow
from pyspark.sql.functions import struct, col

logged_model = "runs:/51a531fb92fb4c0491382b2541460150/model"

# Load model as a Spark UDF.
loaded_model = mlflow.pyfunc.spark_udf(spark, model_uri=logged_model)

# Predict on a Spark DataFrame.
cols = ["person_age", "log_person_income", "loan_percent_income"]
test_pred_df = test_df.withColumn(
    "predictions", loaded_model(struct(*map(col, cols)))[0]
)
display(test_pred_df)

## Real-time Inference (endpoint)

In [0]:
import os
import requests
import numpy as np
import pandas as pd
import json

def create_tf_serving_json(data):
    return {'inputs': {name: data[name].tolist() for name in data.keys()} if isinstance(data, dict) else data.tolist()}

def score_model(dataset):
    url = 'https://adb-34351534749836.16.azuredatabricks.net/serving-endpoints/EL_LoanPrediction_Model_1/invocations'
    headers = {'Authorization': f'Bearer {dbutils.secrets.get(scope="formation-docaposte", key="el_db_api_key")}', 'Content-Type': 'application/json'}
    ds_dict = {'dataframe_split': dataset.to_dict(orient='split')} if isinstance(dataset, pd.DataFrame) else create_tf_serving_json(dataset)
    data_json = json.dumps(ds_dict, allow_nan=True)
    response = requests.request(method='POST', headers=headers, url=url, data=data_json)
    if response.status_code != 200:
        raise Exception(f'Request failed with status {response.status_code}, {response.text}')
    return response.json()

score_model(test_df.select(*feature_cols).limit(100).toPandas())